# API - Medio Aereo Tiempo Real (ultima posición)


## Configuración y librerías

In [41]:
import requests
import time
from datetime import datetime
import datetime as dt
import json
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
from zoneinfo import ZoneInfo
import threading

In [42]:
dataset = '/arcgis/home/config_MedioAereo_Posicionamiento_TiempoReal.json'
with open(dataset, "r", encoding="utf-8") as file:
    config = json.load(file)
    workspace = config.get("workspace")
    capa_helicoptero = config.get("capa_helicoptero")
    medios_aereos = config.get("medios_aereos")
    portal = config.get("portal")
    user = config.get("user")
    password = config.get("password")
    api_key_helicoptero = config.get("api_key_helicoptero")
    api_key_drone = config.get("api_key_drone")
    salidaJson = config.get("salidaJson")
capa_helicoptero
ZONA_HORARIA_CONTROL = ZoneInfo("Atlantic/Canary")
HORA_PARADA_INICIO = dt.time(2, 50)
HORA_PARADA_FIN = dt.time(3, 10)

In [43]:
gis = GIS(portal, user, password, verify_cert=False)
capa = FeatureLayer(capa_helicoptero, gis=gis)

Setting `verify_cert` to False is a security risk, use at your own risk.


## Funciones

In [53]:
def cargar_entidades(data, medio_aereo):
#     with open(json_path, 'r', encoding='utf-8') as f:
#         data = json.load(f)

    print(medio_aereo)

    entidades_add = []
    ids_a_eliminar = []

    # 1. Buscar en la capa todos los registros que coincidan con el medio aéreo (nombre)
    where_clause = f"nombre = '{medio_aereo}'"
    print("CLAUSULAS: " + str(where_clause))

    try:
        query = capa.query(where=where_clause, out_fields="objectid")
        for feat in query.features:
            objectid = feat.attributes.get("objectid")
            if objectid is not None:
                ids_a_eliminar.append(objectid)
    except Exception as e:
        print(f"❌ Error al ejecutar la consulta por nombre: {e}")
        return {"error": str(e)}

    # 2. Procesar los datos del JSON y preparar entidades
    for entrada in data['points']:
        punto_datos = entrada.get('point', {})

        try:
            estado = punto_datos.get('info', {}) or {}
            coord = punto_datos.get('points', {}) or {}
            sensores = punto_datos.get('telemetria', {}) or {}
            fecha = punto_datos.get('time', {}) or {}
            serial = data.get("serial", "")

            x = coord.get('LONGITUDE', "")
            y = coord.get('LATITUDE', "")
            z = coord.get('ALTITUDE', "")

            fecha_evento = dt.datetime(
                year=fecha.get('YEAR', 1),
                month=fecha.get('MONTH', 1),
                day=fecha.get('DAY', 1),
                hour=fecha.get('HOUR', 0),
                minute=fecha.get('MINUTE', 0),
                second=fecha.get('SECOND', 0),
                microsecond=fecha.get('MILLISECOND', 0) * 1000
            )

            atributos = {
                "mission_num": estado.get("MISSION_NUM", ""),
                "pkt_num": estado.get("PKT_NUM", ""),
                "status": estado.get("STATUS", ""),
                "longitude": x,
                "latitude": y,
                "altitude": z,
                "climb": coord.get("CLIMB", ""),
                "fix_mode": coord.get("FIX_MODE", ""),
                "heading": coord.get("HEADING", ""),
                "lidar_distance": coord.get("LIDAR_DISTANCE", None),
                "sat_used": coord.get("SAT_USED", None),
                "sat_vis": coord.get("SAT_VIS", None),
                "pitch": coord.get("PITCH", None),
                "roll": coord.get("ROLL", None),
                "speed": coord.get("SPEED", None),
                "gpio_status": sensores.get("GPIO_STATUS", None),
                "gps_status": sensores.get("GPS_STATUS", None),
                "ir_status": sensores.get("IR_STATUS", None),
                "lidar_status": sensores.get("LIDAR_STATUS", None),
                "rgb_status": sensores.get("RGB_STATUS", None),
                "serial": serial, 
                "fecha": fecha_evento,
                "nombre": medio_aereo
            }

            entidad = {
                "geometry": {
                    "x": x,
                    "y": y,
                    "z": z,
                    "spatialReference": {"wkid": 4326}
                },
                "attributes": atributos
            }

            entidades_add.append(entidad)

        except KeyError as e:
            print(f"❌ Error al procesar punto: {e}")
            return {"error": str(e)}

    # 3. Eliminar anteriores y añadir nuevas entidades
    try:
        print("Entidades a añadir:" + str(entidades_add))
        print("Ids a ELIMINAR: " + str(ids_a_eliminar))
        resultado = capa.edit_features(
            deletes=ids_a_eliminar if ids_a_eliminar else None,
            adds=entidades_add
        )
        return resultado
    except Exception as e:
        print(f"❌ Error al guardar en la capa: {e}")
        return {"error": str(e)}


In [54]:
def make_api_call(serial, token, url, headers, medio_aereo):
    try:
        print(medio_aereo)
        response = requests.get(url, headers=headers)
        timestamp = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        json = response.json()

        print(f"\n[{timestamp}] Status Code: {response.status_code}")
        print(f"Response: {response.json()}")
        cargar_entidades(json, medio_aereo)
        return response.status_code

    except requests.exceptions.RequestException as e:
        print(f"\nRequest error: {e}")
        return None

In [55]:
def continuous_polling(medio_aereo, intervalo=10):
    # Configuration
    serial = medio_aereo
    if "dji" in serial.lower():
        token = api_key_drone
    else:
        token = api_key_helicoptero
    url = f"https://qcdn.hightek.it/v2/{serial}/lastPoint.json"
    headers = {
        "Authorization": f"Bearer {token}"
    }

    print("Starting sequential polling...")
    print(f"URL: {url}")
    print("Press CTRL+C to terminate\n")
    print("Waiting 1 second between calls...")

    try:
        while True:
            status = make_api_call(serial, token, url, headers, medio_aereo)

            # If the call fails, wait 5 seconds before retrying
            if status is None:
                print("Waiting 5 seconds before retrying...")
                time.sleep(intervalo)
                continue

            # Wait 1 second after each response before the next call
            print("\nWaiting 1 second...")
            time.sleep(1)

    except KeyboardInterrupt:
        print("\nPolling terminated by user")

In [56]:
def hora_dentro_de_franja(hora_actual, hora_inicio, hora_fin):
    if hora_inicio <= hora_fin:
        return hora_inicio <= hora_actual <= hora_fin

    return hora_actual >= hora_inicio or hora_actual <= hora_fin


def parar_notebook_si_esta_en_franja():
    """
    Para la ejecución del notebook si la hora actual está
    dentro de la ventana de parada configurada.
    """
    ahora = dt.datetime.now(ZONA_HORARIA_CONTROL)
    hora_actual = ahora.time()

    if hora_dentro_de_franja(
        hora_actual,
        HORA_PARADA_INICIO,
        HORA_PARADA_FIN
    ):
        print(
            f"🛑 Parando notebook de ESRI por ventana horaria de mantenimiento. "
            f"Hora actual: {ahora.strftime('%Y-%m-%d %H:%M:%S %Z')}. "
            f"Franja de parada: "
            f"{HORA_PARADA_INICIO.strftime('%H:%M')} - "
            f"{HORA_PARADA_FIN.strftime('%H:%M')}."
        )

        sys.exit(0)

## Ejecución

In [57]:
import itertools

if __name__ == "__main__":
    print("🚁 Iniciando seguimiento secuencial de medios aéreos...\n")
    print("⏱️ Ejecutando alternadamente cada 10 segundos...\n")

    try:
        # Bucle infinito sobre medios aéreos
        for medio_aereo in itertools.cycle(medios_aereos):

             # Control de parada antes de empezar cada actualización
            parar_notebook_si_esta_en_franja()
            
            print(f"▶️ Procesando medio aéreo: {medio_aereo}")
            
            token_medioAereo = ""
            if "dji" in medio_aereo.lower():
                token_medioAereo = api_key_drone
            else:
                token_medioAereo = api_key_helicoptero

            try:
                status = make_api_call(
                    serial=medio_aereo,
                    token=token_medioAereo,
                    url=f"https://qcdn.hightek.it/v2/{medio_aereo}/lastPoint.json",
                    headers={"Authorization": f"Bearer {token_medioAereo}"},
                    medio_aereo=medio_aereo
                )

                if status is None:
                    print(f"[{medio_aereo}] ⚠️ Error controlado (status=None). Saltando al siguiente...\n")
                else:
                    print(f"[{medio_aereo}] ✅ Consulta completada.\n")

            except Exception as e:
                # Captura cualquier fallo de red, timeout, JSON, etc.
                print(f"[{medio_aereo}] ❌ Error en la petición: {e}")
                print("➡️ Continuando con el siguiente medio...\n")

            # Pausa común entre ciclos
            time.sleep(5)

    except KeyboardInterrupt:
        print("\n🛑 Seguimiento detenido por el usuario.")



🚁 Iniciando seguimiento secuencial de medios aéreos...

⏱️ Ejecutando alternadamente cada 10 segundos...

▶️ Procesando medio aéreo: LI-52
LI-52

[2026-06-09 06:47:00] Status Code: 200
Response: {'points': [{'point': {'info': {'MISSION_NUM': 1780941249500, 'PKT_NUM': 2237, 'STATUS': 'FLY'}, 'points': {'ALTITUDE': 56.62885665893555, 'CLIMB': 0.06775959581136703, 'FIX_MODE': 3, 'HEADING': 222.96493530273438, 'LATITUDE': 28.620075225830078, 'LIDAR_DISTANCE': -1, 'LONGITUDE': -17.753599166870117, 'PITCH': -1.009886622428894, 'ROLL': -158.44207763671875, 'SAT_USED': 16, 'SPEED': 0.18383018672466278, 'TKF_ALTITUDE': 813.853759765625, 'YAW': 117.26990509033203}, 'telemetria': {'GPIO_STATUS': 1, 'IR_STATUS': 1, 'LIDAR_STATUS': 0, 'RGB_STATUS': 1}, 'time': {'DAY': 8, 'HOUR': 18, 'MILLISECOND': 21, 'MINUTE': 42, 'MONTH': 6, 'SECOND': 25, 'YEAR': 2026}, 'epoch': 1780944145}}]}
LI-52
CLAUSULAS: nombre = 'LI-52'
Entidades a añadir:[{'geometry': {'x': -17.753599166870117, 'y': 28.620075225830078, 'z